# Phase 2 — Step 7: SHAP Explainability Analysis

**Best models from Step 6 grid search:**
- RF : n_estimators=200, max_depth=6, CV AUC=0.9052
- XGB: n_estimators=100, max_depth=4, CV AUC=0.9056

**What this notebook does:**
1. Loads best models from Drive
2. SHAP scatter plot (replaces beeswarm — works with single feature)
3. SHAP bar plot — mean absolute importance
4. SHAP force plot — explains highest risk variant
5. SHAP waterfall plot — shows base vs final prediction
6. SHAP on all 22 Phase 1 candidate variants
7. Saves all figures + CSV to Drive

**Run cells one by one**

In [ ]:
!pip install shap scikit-learn xgboost pandas numpy matplotlib joblib -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
import os
warnings.filterwarnings('ignore')
os.makedirs('figures_step7', exist_ok=True)
print(f'SHAP version: {shap.__version__}')
print('Libraries ready')

In [ ]:
from google.colab import drive
import joblib, os
drive.mount('/content/drive')

RF_PATH  = '/content/drive/My Drive/best_rf_model_step6.pkl'
XGB_PATH = '/content/drive/My Drive/best_xgb_model_step6.pkl'
IMP_PATH = '/content/drive/My Drive/imputer_step6.pkl'
TRAIN_PATH = '/content/drive/My Drive/clinvar_training_data_v2.csv'
TEST_PATH  = '/content/drive/My Drive/all_variants_enriched_step3.csv'

for p in [RF_PATH, XGB_PATH, IMP_PATH, TRAIN_PATH, TEST_PATH]:
    status = '✅' if os.path.exists(p) else '❌'
    print(f'{status} {os.path.basename(p)}')

best_rf  = joblib.load(RF_PATH)
best_xgb = joblib.load(XGB_PATH)
imputer  = joblib.load(IMP_PATH)
print('Models loaded')

import pandas as pd
train_df = pd.read_csv(TRAIN_PATH)
var_df   = pd.read_csv(TEST_PATH, low_memory=False)
print(f'Training: {len(train_df):,} | Test: {len(var_df):,}')

In [ ]:
import numpy as np

FEATURE_COLS  = ['cadd_phred']
FEATURE_LABEL = 'CADD Phred Score'

X_train = imputer.transform(train_df[FEATURE_COLS].values)
y_train = train_df['label'].values.astype(int)
X_test  = imputer.transform(var_df[FEATURE_COLS].values)

# Sample 3000 for SHAP plots
np.random.seed(42)
idx = np.random.choice(len(X_test), size=3000, replace=False)
X_shap = X_test[idx]

# Background for explainer
bg_idx = np.random.choice(len(X_train), size=500, replace=False)
X_bg   = X_train[bg_idx]

print(f'SHAP sample : {X_shap.shape}')
print(f'Background  : {X_bg.shape}')
print(f'CADD range  : {X_shap[:,0].min():.2f} — {X_shap[:,0].max():.2f}')

In [ ]:
# RF SHAP values
print('Computing RF SHAP values...')
rf_explainer = shap.TreeExplainer(best_rf, data=X_bg,
                                   feature_perturbation='interventional')
rf_shap_raw = rf_explainer.shap_values(X_shap)

# rf_shap_raw is (3000, 1, 2) — take class 1 (pathogenic), flatten
if isinstance(rf_shap_raw, list):
    rf_sv = rf_shap_raw[1].flatten()
elif rf_shap_raw.ndim == 3:
    rf_sv = rf_shap_raw[:, 0, 1]
else:
    rf_sv = rf_shap_raw.flatten()

rf_base = rf_explainer.expected_value
if isinstance(rf_base, (list, np.ndarray)):
    rf_base = float(rf_base[1])
else:
    rf_base = float(rf_base)

print(f'RF SHAP shape : {rf_sv.shape}')
print(f'RF base value : {rf_base:.4f}')
print(f'Mean |SHAP|   : {np.abs(rf_sv).mean():.4f}')

In [ ]:
# SHAP Scatter Plot (replaces beeswarm for single feature)
cadd_vals = X_shap[:, 0]

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(cadd_vals, rf_sv, c=cadd_vals, cmap='RdBu_r',
                alpha=0.4, s=8, edgecolors='none')
plt.colorbar(sc, ax=ax, label='CADD Phred Value')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('CADD Phred Score', fontsize=12)
ax.set_ylabel('SHAP Value', fontsize=12)
ax.set_title('SHAP Analysis — Random Forest\n'
             'How CADD Phred drives pathogenicity predictions',
             fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/rf_shap_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('RF SHAP scatter saved')

In [ ]:
# SHAP Bar Plot
mean_abs = np.abs(rf_sv).mean()

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh([FEATURE_LABEL], [mean_abs], color='#C0392B', height=0.4)
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('SHAP Feature Importance — Random Forest\n'
             'Mean absolute SHAP value (cadd_phred)',
             fontsize=12, fontweight='bold')
ax.text(mean_abs+0.001, 0, f'{mean_abs:.4f}', va='center', fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/rf_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'RF SHAP bar saved — Mean |SHAP| = {mean_abs:.4f}')

In [ ]:
# Force + Waterfall for highest risk variant
rf_probs = best_rf.predict_proba(X_shap)[:, 1]
top_idx  = np.argmax(rf_probs)
top_prob = rf_probs[top_idx]
top_cadd = float(X_shap[top_idx, 0])
top_shap = float(rf_sv[top_idx])

print(f'Highest risk variant: CADD={top_cadd:.2f}, P={top_prob:.4f}, SHAP={top_shap:.4f}')

# Force plot
shap.force_plot(
    rf_base,
    np.array([[top_shap]]),
    np.array([[top_cadd]]),
    feature_names=[FEATURE_LABEL],
    matplotlib=True, show=False, figsize=(12, 3)
)
plt.title(f'SHAP Force Plot — RF | CADD={top_cadd:.1f}, P={top_prob:.3f}',
          fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('figures_step7/rf_shap_force.png', dpi=150, bbox_inches='tight')
plt.show()
print('RF Force plot saved')

In [ ]:
# Waterfall plot (manual — SHAP waterfall broken for single feature)
final = rf_base + top_shap

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([FEATURE_LABEL], [top_shap], left=[rf_base],
        color='#C0392B' if top_shap > 0 else '#2E5FA3',
        height=0.4, edgecolor='white')
ax.axvline(rf_base, color='grey', ls='--', lw=1.5,
           label=f'Base value = {rf_base:.3f}')
ax.axvline(final,   color='black', ls='-', lw=2,
           label=f'Final P = {final:.3f}')
ax.set_xlabel('Prediction Probability', fontsize=11)
ax.set_title(f'SHAP Waterfall Plot — RF\n'
             f'CADD={top_cadd:.1f} | Base={rf_base:.3f} → Final={final:.3f}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/rf_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('RF Waterfall saved')

In [ ]:
# XGBoost SHAP
print('Computing XGB SHAP values...')
xgb_explainer = shap.TreeExplainer(best_xgb)
xgb_shap_raw  = xgb_explainer.shap_values(X_shap)

if isinstance(xgb_shap_raw, list):
    xgb_sv = xgb_shap_raw[1].flatten()
elif xgb_shap_raw.ndim == 3:
    xgb_sv = xgb_shap_raw[:, 0, 1]
else:
    xgb_sv = xgb_shap_raw.flatten()

xgb_base = float(xgb_explainer.expected_value)
print(f'XGB SHAP shape: {xgb_sv.shape}')
print(f'XGB base value: {xgb_base:.4f}')

# XGB Scatter
fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(cadd_vals, xgb_sv, c=cadd_vals, cmap='RdBu_r',
                alpha=0.4, s=8, edgecolors='none')
plt.colorbar(sc, ax=ax, label='CADD Phred Value')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('CADD Phred Score', fontsize=12)
ax.set_ylabel('SHAP Value', fontsize=12)
ax.set_title('SHAP Analysis — XGBoost\n'
             'How CADD Phred drives pathogenicity predictions',
             fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/xgb_shap_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('XGB SHAP scatter saved')

In [ ]:
# XGB Bar
xgb_mean_abs = np.abs(xgb_sv).mean()
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh([FEATURE_LABEL], [xgb_mean_abs], color='#C0392B', height=0.4)
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('SHAP Feature Importance — XGBoost\n'
             'Mean absolute SHAP value (cadd_phred)',
             fontsize=12, fontweight='bold')
ax.text(xgb_mean_abs+0.001, 0, f'{xgb_mean_abs:.4f}', va='center', fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/xgb_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'XGB SHAP bar saved — Mean |SHAP| = {xgb_mean_abs:.4f}')

# XGB Force + Waterfall
xgb_probs   = best_xgb.predict_proba(X_shap)[:, 1]
xgb_top_idx = np.argmax(xgb_probs)
xgb_top_prob = xgb_probs[xgb_top_idx]
xgb_top_cadd = float(X_shap[xgb_top_idx, 0])
xgb_top_shap = float(xgb_sv[xgb_top_idx])

shap.force_plot(
    xgb_base,
    np.array([[xgb_top_shap]]),
    np.array([[xgb_top_cadd]]),
    feature_names=[FEATURE_LABEL],
    matplotlib=True, show=False, figsize=(12, 3)
)
plt.title(f'SHAP Force Plot — XGB | CADD={xgb_top_cadd:.1f}, P={xgb_top_prob:.3f}',
          fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('figures_step7/xgb_shap_force.png', dpi=150, bbox_inches='tight')
plt.show()
print('XGB Force saved')

# XGB Waterfall
xgb_final = xgb_base + xgb_top_shap
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([FEATURE_LABEL], [xgb_top_shap], left=[xgb_base],
        color='#C0392B' if xgb_top_shap > 0 else '#2E5FA3', height=0.4)
ax.axvline(xgb_base,  color='grey',  ls='--', lw=1.5, label=f'Base={xgb_base:.3f}')
ax.axvline(xgb_final, color='black', ls='-',  lw=2,   label=f'Final={xgb_final:.3f}')
ax.set_xlabel('Prediction (log-odds)', fontsize=11)
ax.set_title(f'SHAP Waterfall — XGB | CADD={xgb_top_cadd:.1f}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/xgb_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('XGB Waterfall saved')

In [ ]:
# SHAP on 22 Phase 1 Candidate Variants
import pandas as pd
import numpy as np

VARIANTS_22 = pd.DataFrame({
    'gene'       : ['HP1BP3','AGAP4','KDM5A','C12orf40','PAPOLA','RTF1','USP7',
                    'EPG5','PIGF','SPICE1','SPICE1','SLC2A2','ZNF141','TRIML2',
                    'TRIML2','SREK1','SREK1','RAD50','RARS2','TNRC18','DBF4','LRRCC1'],
    'impact'     : ['MODERATE','MODERATE','MODERATE','MODERATE','HIGH','MODERATE',
                    'MODERATE','MODERATE','MODERATE','HIGH','HIGH','MODERATE',
                    'MODERATE','MODERATE','MODERATE','MODERATE','MODERATE','MODERATE',
                    'MODERATE','HIGH','MODERATE','MODERATE'],
    'cadd_phred' : [23.5,4.63,13.88,3.54,2.71,22.0,24.7,21.1,0.43,
                   25.1,25.8,0.0,1.66,9.62,19.43,23.7,23.0,1.07,
                   1.45,22.2,0.57,5.45],
})

X_22 = imputer.transform(VARIANTS_22[['cadd_phred']].values)
rf_probs_22  = best_rf.predict_proba(X_22)[:, 1]
xgb_probs_22 = best_xgb.predict_proba(X_22)[:, 1]
rf_sv_22     = rf_explainer.shap_values(X_22)

if isinstance(rf_sv_22, list):
    rf_sv_22 = rf_sv_22[1].flatten()
elif rf_sv_22.ndim == 3:
    rf_sv_22 = rf_sv_22[:, 0, 1]
else:
    rf_sv_22 = rf_sv_22.flatten()

VARIANTS_22['rf_prob']  = np.round(rf_probs_22, 4)
VARIANTS_22['xgb_prob'] = np.round(xgb_probs_22, 4)
VARIANTS_22['consensus']= np.round((rf_probs_22+xgb_probs_22)/2, 4)
VARIANTS_22['rf_shap']  = np.round(rf_sv_22, 4)
VARIANTS_22['tier']     = ['HIGH' if p>=0.75 else 'MODERATE' if p>=0.50 else 'LOW'
                           for p in VARIANTS_22['consensus']]

print('22 CANDIDATE VARIANTS — SHAP + PREDICTIONS')
print('='*80)
print(VARIANTS_22[['gene','impact','cadd_phred','rf_prob','xgb_prob',
                    'consensus','tier','rf_shap']].to_string(index=False))

In [ ]:
# Plot SHAP for 22 variants
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

tier_colors = {'HIGH':'#C0392B','MODERATE':'#E67E22','LOW':'#27AE60'}
colors = [tier_colors[t] for t in VARIANTS_22['tier']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(VARIANTS_22['gene'], VARIANTS_22['rf_shap'],
               color=colors, height=0.65, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('SHAP Value (contribution to pathogenicity)', fontsize=11)
ax.set_title('SHAP Values — 22 Phase 1 Candidate Variants\n'
             'Random Forest | Real CADD Scores',
             fontsize=12, fontweight='bold')

for bar, cadd in zip(bars, VARIANTS_22['cadd_phred']):
    xpos = bar.get_width() + 0.001
    ax.text(xpos, bar.get_y()+bar.get_height()/2,
            f'CADD={cadd:.1f}', va='center', fontsize=7)

legend_els = [mpatches.Patch(fc='#C0392B', label='HIGH risk'),
              mpatches.Patch(fc='#E67E22', label='MODERATE risk'),
              mpatches.Patch(fc='#27AE60', label='LOW risk')]
ax.legend(handles=legend_els, fontsize=9)
ax.invert_yaxis()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step7/shap_22_candidates.png', dpi=150, bbox_inches='tight')
plt.show()
print('22-variant SHAP figure saved')

In [ ]:
import shutil
from google.colab import files

# Save CSV to Drive
csv_path = '/content/drive/My Drive/shap_22_candidates_step7.csv'
VARIANTS_22.to_csv(csv_path, index=False)
print(f'CSV saved to Drive: {csv_path}')

# List all figures
figs = sorted(os.listdir('figures_step7'))
print(f'Figures saved ({len(figs)}):')
for f in figs:
    print(f'  {f}')

# Zip and download
shutil.make_archive('step7_outputs', 'zip', 'figures_step7')
files.download('step7_outputs.zip')
print('Downloaded step7_outputs.zip')